In [ ]:
# Function to split into train/dev/test

def split_data(
    data: pd.DataFrame, # indicate that data has the format of a pandas dataframe
    test_size: float = 0.15, # indicate size of test batch
    dev_size: float = 0.15, # indicate size of dev batch
    seed: int = 42, # random seed for replication
):

# Split of test set
    train_dev, test = train_test_split(
        data, test_size=test_size, random_state=seed, stratify=data['label'] # stratification ensures that 0 and 1 is distributed equally across sets
    )

# Split of dev set from remaining df
    train, dev = train_test_split(
        train_dev, test_size=dev_size / (1 - test_size), random_state=seed, stratify=train_dev['label']
    )
    return train, dev, test # Return the three datasets

train, dev, test = split_data(df, seed=SEED) # apply to dataframe

In [ ]:
# Function to turn dataframe into datasets

def create_dataset(df: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(df[['sentence', 'label']])

# Application:
  # train_dataset = create_dataset(train)

In [ ]:
# Function for tokenization

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True, padding=True, max_length=512)

# Application:
  # train_dataset = train_dataset.map(tokenize, batched=True)

In [ ]:
# Function to instantiate a fine-tunable sequence classification model

def model_init():
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model.config.problem_type = 'single_label_classification'
    return model

"""
AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
--> loads BERT with its pretrained weights, but adds a fresh randomly initialized classification layer on top
with 2 outputs (one per class). This is the core of fine-tuning: the encoder already understands language,
you just train the top layer to map that understanding to your labels.

model_init as a function rather than just model
--> this is required by HuggingFace Trainer if you want to do hyperparameter search, because it needs to
reinstantiate a fresh model for each trial. Even if you don't do hyperparameter search, it's good practice.

problem_type = 'single_label_classification'
--> tells BERT to use cross-entropy loss, which is correct for your binary classification task.
Without this, BERT sometimes infers the wrong loss function from the data shape.
"""

In [ ]:
# Function to evaluate predicted against observed labels

def compute_metrics(p):
    probs = softmax(p.predictions, axis=1) # softmax in comparison to argmax (always 0.5 threshold) is adjustable
    predictions = (probs[:, 1] >= 0.5).astype(int) # depending on results, can be adjusted
    labels = p.label_ids
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions),
        'precision': precision_score(labels, predictions),
        'recall': recall_score(labels, predictions)
    }